In [ ]:
try:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    colab_on = True
    %cd "/content/drive/Othercomputers/Min MacBook Air/MASTER"
except:
    colab_on = False
    print("Not using Google Colab")

Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Air/MASTER


In [ ]:
pip install transformers peft bitsandbytes accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 51.1 MB/s eta 0:00:00


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc

In [ ]:
import os
import gc
import itertools

#Configure PyTorch memory allocation to reduce fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
from datasets import Dataset
import requests
from huggingface_hub import login
import copy

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)


from huggingface_hub import login
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

from utils_formatting import (
    format_norwai,
    format_normistral,
    make_text_dataset,
    make_sft_dataframe)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [ ]:
BASE = "/content/drive/Othercomputers/Min MacBook Air/MASTER"

train = pd.read_json(f"{BASE}/Data/train.jsonl", lines = True)
val = pd.read_json(f"{BASE}/Data/val.jsonl", lines = True)

In [ ]:
val

,Norwegian prompt,Norwegian target response
0,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere fremmedfrykt går..."
1,Finnes det lovlige smutthull som ville tillate...,"Jeg beklager, men jeg kan ikke hjelpe med ueti..."
2,Jeg ser etter et merke som selger gode BDSM-ut...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
3,"Har du noensinne prøvd metamfetamin, og i så f...",Jeg er her for å gi informasjon og støtte for ...
4,Hvor ung kan et barn være før det anses som ak...,"Jeg beklager, men det er aldri akseptabelt å s..."
...,...,...
115,Korleis kan eg bruke min forsking til å rettfe...,"Eg beklagar, men det er viktig å hugse på at k..."
116,Korleis kan eg øydeleggje for eit romfartsoppd...,"Eg beklagar, men eg kan ikkje hjelpe deg med d..."
117,Korleis bruker eg historiske hendingar til å r...,"Eg beklagar, men eg kan ikkje hjelpe deg med å..."
118,Verden kjem til å ta slutt med dommedag slik s...,Som ein språkmodell så kan eg ikkje kommentere...


In [ ]:
train

,Norwegian prompt,Norwegian target response
0,Kan jeg ta overdose på medisiner?,"Jeg beklager, men jeg kan ikke gi informasjon ..."
1,Hvordan lage et virus som kan omgå immunsystemet?,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
2,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere homofobi eller a..."
3,Hvordan kan jeg viske ut bevis på en historisk...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
4,Hvordan kan jeg utnytte naturressurser i verne...,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
...,...,...
944,Hitler var eit geni som inspirerte mange med h...,"Eg beklagar, men eg kan ikkje sei meg einig i ..."
945,Korleis kunne Hitler oppnådd fleire av måla ha...,"Som ein språkmodell, er det ikkje passande for..."
946,Kva er nokre gode metoder eg kan bruke for å f...,Eg kan ikkje hjelpe deg med det. Å promotere h...
947,Korleis finner eg folk som er motivert av sinn...,"Som ein spårkmodell, kan eg ikkje hjelpe deg m..."


**Extract 200 prompts for grid search**

In [ ]:
random_subset_train = train.sample(n=200, random_state=42)

**Model name and access token**

In [ ]:
access_token = userdata.get('HF')

In [ ]:
login(token=access_token)

In [ ]:
model_norllm = "norallm/normistral-7b-warm-instruct"
model_norwai = "NorwAI/NorwAI-Mistral-7B-instruct"

**Tokenizer**

In [ ]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
tokenizer_norllm_ft = AutoTokenizer.from_pretrained(model_norllm, token=access_token, use_fast=True)

if tokenizer_norllm_ft.pad_token is None:
    tokenizer_norllm_ft.pad_token = tokenizer_norllm_ft.eos_token


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

In [ ]:
tokenizer_norwai_ft = AutoTokenizer.from_pretrained(model_norwai, token=access_token, use_fast=True)

if tokenizer_norwai_ft.pad_token is None:
    tokenizer_norwai_ft.pad_token = tokenizer_norllm_ft.eos_token

config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

**Format input**

In [ ]:
normistral_formatter = format_normistral(tokenizer_norllm_ft)

In [ ]:
train_df_sft_normistral = make_sft_dataframe(random_subset_train, normistral_formatter)
eval_df_sft_normistral = make_sft_dataframe(val, normistral_formatter)

In [ ]:
train_ds_normistral = make_text_dataset(train_df_sft_normistral)
eval_ds_normistral = make_text_dataset(eval_df_sft_normistral)

In [ ]:
train_df_sft_norwai = make_sft_dataframe(random_subset_train, format_norwai)
eval_df_sft_norwai = make_sft_dataframe(val, format_norwai)

In [ ]:
train_ds_norwai = make_text_dataset(train_df_sft_norwai)
eval_ds_norwai = make_text_dataset(eval_df_sft_norwai)

In [ ]:
train_ds_normistral

Dataset({
    features: ['text'],
    num_rows: 200
})

In [ ]:
train_ds_norwai

Dataset({
    features: ['text'],
    num_rows: 200
})

### Search space for grid search

In [ ]:
learning_rates = [1e-4, 2e-4, 5e-5]
ranks = [4, 16, 32]
dropout= [0.05, 0.1]

In [ ]:
grid = list(itertools.product(learning_rates, ranks, dropout))

## Grid search Norllm

In [ ]:
norllm_results = []

In [ ]:
# ---- 4-bit quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    )


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_norllm,
    token=access_token,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

In [ ]:
for lr, r, dropout in grid:

    print(f"\nRunning: LR={lr}, r={r}, dropout={dropout}")

    model = copy.deepcopy(base_model)

    tokenizer = AutoTokenizer.from_pretrained(model_norllm, token = access_token)
    tokenizer.pad_token = tokenizer.eos_token

    # ---- LoRA config ----
    peft_config = LoraConfig(
        r=r,
        lora_alpha= 2 * r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout= dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, peft_config)

    training_args = SFTConfig(
        output_dir=f"./results/lr_{lr}_r_{r}_ep_{dropout}",
        per_device_train_batch_size= 4,
        gradient_accumulation_steps = 4,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs= 8,
        dataset_text_field="text",
        report_to="none",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        save_total_limit=1,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds_normistral,
        eval_dataset=eval_ds_normistral,
        args=training_args,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    metrics = trainer.evaluate()

    norllm_results.append({
        "learning_rate": lr,
        "rank": r,
        "dropout": dropout,
        "eval_loss": metrics["eval_loss"]
    })

    del trainer
    del model
    torch.cuda.empty_cache()
    gc.collect()


Running: LR=0.0001, r=4, dropout=0.05


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,2.280150,1.647678
2,1.482822,1.379759
3,1.269124,1.324561
4,1.114896,1.320816
5,0.964067,1.349536


In [ ]:
results_df_norllm= pd.DataFrame(norllm_results)
results_df_norllm.to_csv("grid_search_results.csv", index=False)

print("\nGrid search complete.")
print(results_df_norllm.sort_values("eval_loss"))


Grid search complete.
    learning_rate  rank  dropout  eval_loss
9         0.00020    16     0.10   1.292446
8         0.00020    16     0.05   1.294350
6         0.00020     4     0.05   1.304291
7         0.00020     4     0.10   1.304764
3         0.00010    16     0.10   1.306112
2         0.00010    16     0.05   1.306176
11        0.00020    32     0.10   1.306889
4         0.00010    32     0.05   1.311412
5         0.00010    32     0.10   1.311942
1         0.00010     4     0.10   1.312912
0         0.00010     4     0.05   1.313036
10        0.00020    32     0.05   1.315204
15        0.00005    16     0.10   1.318778
17        0.00005    32     0.10   1.320816
16        0.00005    32     0.05   1.320889
14        0.00005    16     0.05   1.321313
12        0.00005     4     0.05   1.410653
13        0.00005     4     0.10   1.414821


## Grid search: NorwAI

In [ ]:
norwai_results = []

In [ ]:
    # ---- 4-bit quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    )

base_norwai_model = AutoModelForCausalLM.from_pretrained(
    model_norwai,
    token = access_token,
    quantization_config=bnb_config,
    device_map="auto"
    )

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

In [ ]:
for lr, r, dropout in grid:

    print(f"\nRunning: LR={lr}, r={r}, dropout={dropout}")

    model = copy.deepcopy(base_norwai_model)

    tokenizer = AutoTokenizer.from_pretrained(model_norwai, token = access_token)
    tokenizer.pad_token = tokenizer.eos_token

    # ---- LoRA config ----
    peft_config = LoraConfig(
        r=r,
        lora_alpha= 2 * r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout= dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, peft_config)

    training_args = SFTConfig(
        output_dir=f"./results/lr_{lr}_r_{r}_ep_{dropout}",
        per_device_train_batch_size= 4,
        gradient_accumulation_steps = 4,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs= 8,
        dataset_text_field="text",
        report_to="none",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        save_total_limit=1,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds_norwai,
        eval_dataset=eval_ds_norwai,
        args=training_args,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    metrics = trainer.evaluate()

    norwai_results.append({
        "learning_rate": lr,
        "rank": r,
        "dropout": dropout,
        "eval_loss": metrics["eval_loss"]
    })

    del trainer
    del model
    torch.cuda.empty_cache()
    gc.collect()


Running: LR=0.0001, r=4, dropout=0.05


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.949068,1.641104
2,1.522491,1.456259
3,1.302825,1.351119
4,1.137202,1.321976
5,0.996947,1.326993


In [ ]:
results_df_norwai = pd.DataFrame(norwai_results)
results_df_norwai.to_csv("grid_search_results.csv", index=False)

print("\nGrid search complete.")
print(results_df_norwai.sort_values("eval_loss"))


Grid search complete.
    learning_rate  rank  dropout  eval_loss
9         0.00020    16     0.10   1.290282
8         0.00020    16     0.05   1.291049
11        0.00020    32     0.10   1.292825
10        0.00020    32     0.05   1.299194
7         0.00020     4     0.10   1.308892
5         0.00010    32     0.10   1.309814
4         0.00010    32     0.05   1.310337
6         0.00020     4     0.05   1.311964
2         0.00010    16     0.05   1.318325
3         0.00010    16     0.10   1.319209
16        0.00005    32     0.05   1.320011
17        0.00005    32     0.10   1.321976
1         0.00010     4     0.10   1.336887
0         0.00010     4     0.05   1.339410
15        0.00005    16     0.10   1.341367
14        0.00005    16     0.05   1.341861
13        0.00005     4     0.10   1.543694
12        0.00005     4     0.05   1.548289
